# Stage 0 — pdfplumber Text Extraction

**Why this approach:**  
The original pypdf approach (see `stage0_exploration.ipynb`) had three problems:
1. Header/footer text (company name, date, page number) injected mid-sentence across page boundaries
2. No paragraph spacing preserved within a speaker's comment
3. Analyst names missed by regex because pypdf loses column layout

**This approach:**
1. `pdfplumber` with `layout=True` — preserves paragraph spacing, better text ordering
2. Strip company name / date / page number per page before concatenating (they appear as standalone lines)
3. Detect all speaker names from `Name:` patterns in the clean text — no coordinate tracking
4. Parse management names from page 2 participant list for role classification
5. Split full text by speaker names → ordered `(speaker, role, comment)` turns

## Setup

In [223]:
import re
import pdfplumber
from collections import Counter

PDF_PATH = r"../transcripts/fineotex_chemical_Q4_FY26.pdf"
# PDF_PATH = r"../transcripts/asian_paints_Q4_FY26.pdf"
# PDF_PATH = r"../transcripts/sandhar_technologies_Q4_FY26.pdf"
# PDF_PATH = r"../transcripts/mold-tek_packaging_Q4_FY26.pdf"

pdf = pdfplumber.open(PDF_PATH)
print(f"PDF   : {PDF_PATH}")
print(f"Pages : {len(pdf.pages)}")

PDF   : ../transcripts/fineotex_chemical_Q4_FY26.pdf
Pages : 21


---
## Step 1 — Extract management and moderator names from page 2

Page 2 always has a participant list. Format varies by transcript:
- Fineotex: `MS. AARTI JHUNJHUNWALA – EXECUTIVE DIRECTOR`  (ALL-CAPS, dash separator)
- Asian Paints: `Mr. Amit Syngle : MD & CEO`  (Title Case, colon separator)

Some transcripts also list the sell-side moderator (e.g. Mold-Tek lists Rajesh Kumar under `MODERATOR:`).  
Both blocks use the same `honorific + name + separator` pattern.  
Names normalized to Title Case for consistent matching later.

In [224]:
# Inspect raw page 2 text to understand the participant list format
page2_text = pdf.pages[1].extract_text(layout=True) or ""
print(page2_text)

                                                                                  
                                                                                  
                                                                                  
                                                                                  
                                                                                  
                                                                                  
                                                                                  
                                                                                  
                                                                                  
                                                                                  
                                                                                  
                         “Fineotex   Chemical   Limited                           
    

In [225]:
# Matches: honorific + name (ALL-CAPS or Title Case) + separator (dash or colon)
# Handles both:
#   MS. AARTI JHUNJHUNWALA – EXECUTIVE DIRECTOR   (Fineotex — ALL-CAPS, dash)
#   Mr. Amit Syngle : MD & CEO                    (Asian Paints — Title Case, colon)
_MGMT_NAME_RE = re.compile(
    r"(?:MR|MS|MRS|DR)\.\s+([A-Z][A-Za-z.]+(?:\s+[A-Z][A-Za-z.]+){0,2})\s*[–\-:]",
    re.IGNORECASE
)


def _extract_names_from_block(page2_text, block_label):
    """Extract honorific-prefixed names from a named block (e.g. MANAGEMENT or MODERATOR).

    Reads from 'BLOCK_LABEL: ...' up to the next block label or end of text.
    Returns a set of Title Case names.
    """
    # Match the block label and capture everything after it
    m = re.search(
        rf"^[ \t]*{re.escape(block_label)}\s*:(.*?)(?=^[ \t]*(?:MANAGEMENT|MODERATOR|ANALYST|PARTICIPANTS?|SPEAKERS?)\s*:|\Z)",
        page2_text,
        re.S | re.I | re.MULTILINE,
    )
    if not m:
        return set()
    block_text = m.group(1)
    names = set()
    for raw in _MGMT_NAME_RE.findall(block_text):
        name = " ".join(raw.split()).title()   # collapse whitespace + Title Case
        names.add(name)
    return names


def extract_management_names(page2_text):
    # Try common block labels used across different transcription services
    for label in ("MANAGEMENT", "PARTICIPANTS", "PARTICIPANT", "SPEAKERS", "SPEAKER"):
        names = _extract_names_from_block(page2_text, label)
        if names:
            return names
    print("WARNING: No management block found on page 2 — all non-moderator speakers will be labelled analyst")
    return set()


def extract_moderator_names(page2_text):
    """Extract named moderators listed under MODERATOR: on page 2 (e.g. sell-side host).

    These are distinct from the generic 'Moderator' speaker tag used by the transcription service.
    If no MODERATOR block exists (most transcripts), returns an empty set — not a problem.
    """
    return _extract_names_from_block(page2_text, "MODERATOR")


mgmt_names = extract_management_names(page2_text)
moderator_names = extract_moderator_names(page2_text)

print(f"Management names ({len(mgmt_names)}):")
for name in sorted(mgmt_names):
    print(f"  {name}")

print()
print(f"Moderator names from roster ({len(moderator_names)}):")
for name in sorted(moderator_names):
    print(f"  {name}")
if not moderator_names:
    print("  (none listed — 'Moderator' keyword used in transcript body)")

Management names (4):
  Aarti Jhunjhunwala
  Arindam Choudhuri
  Sanjay Tibrewala
  Yusuf Contractor

Moderator names from roster (0):
  (none listed — 'Moderator' keyword used in transcript body)


---
## Step 2 — Extract raw text from content pages

`extract_text(layout=True)` preserves paragraph gaps as blank lines and keeps indentation.  
Content starts at page 3 (pages 1–2 are cover letter and participant list).  

Inspect raw output before any cleaning.

In [226]:
CONTENT_START_PAGE = 3   # 1-indexed — first page with speaker turns

raw_pages = []
for page in pdf.pages[CONTENT_START_PAGE - 1:]:
    text = page.extract_text(layout=True) or ""
    raw_pages.append(text)

print(f"Content pages: {len(raw_pages)}")
print(f"Total chars  : {sum(len(p) for p in raw_pages):,}")

Content pages: 19
Total chars  : 118,363


In [227]:
# Inspect one raw page — look at header at the top and footer at the bottom
PAGE_TO_INSPECT = 0   # 0 = first content page

print(f"=== Raw page {CONTENT_START_PAGE + PAGE_TO_INSPECT} (before cleaning) ===")
print(raw_pages[PAGE_TO_INSPECT])

=== Raw page 3 (before cleaning) ===
                                                                                  
                                                                                  
                                                                                  
                                                         Fineotex Chemical Limited
                                                                May 18, 2026      
                                                                                  
                                                                                  
                                                                                  
                                                                                  
          Moderator:     Ladies and gentlemen, good day, and welcome to the Fineotex Chemical Limited Q4 and FY '26
                         Earnings Conference Call. As a reminder, all participant lines will be in t

---
## Step 3 — Strip header / footer from each page

The header (company name + date) always appears at the **top** of each page, before any speaker content.  
The footer (page number) always appears at the **bottom** of each page, after all content.

**Approach:** scan inward from each edge — skip blank lines, blank out header/footer lines, stop the moment real content is reached. Never touch lines in the middle of the page.

In [228]:
_MONTHS = (
    "January|February|March|April|May|June|"
    "July|August|September|October|November|December"
)
_DATE_RE        = re.compile(rf"^({_MONTHS})\s+\d{{1,2}},?\s+\d{{4}}$", re.I)
_COMPANY_RE     = re.compile(r"^.+\b(Limited|Ltd\.?|Inc\.?|Corp\.?|Pvt\.?)$", re.I)
_PAGE_NUMBER_RE = re.compile(r"^Page\s+\d+\s+of\s+\d+$", re.I)
_PAGE_PIPE_RE   = re.compile(r"^\d+\s*\|$|^\|\s*\d+$")   # matches "2 |" or "| 2" only


def _is_header_footer_line(s):
    """True if stripped line is a company name, date, or page number."""
    return bool(_DATE_RE.match(s) or _COMPANY_RE.match(s) or _PAGE_NUMBER_RE.match(s))


def strip_header_footer(page_text):
    lines = page_text.split("\n")

    # Scan from the TOP: skip blanks, blank out header lines, stop at first real content
    for i, line in enumerate(lines):
        s = line.strip()
        if not s:
            continue                    # blank line — keep scanning
        if _is_header_footer_line(s):
            lines[i] = ""              # header line — remove it
        else:
            break                       # real content starts here — stop

    # Scan from the BOTTOM: skip blanks, blank out footer lines, stop at first real content
    checked_last = False   # tracks whether we've examined the last non-empty line
    for i in range(len(lines) - 1, -1, -1):
        s = lines[i].strip()
        if not s:
            continue                    # blank line — keep scanning
        if not checked_last:
            checked_last = True
            # pipe-style page numbers (e.g. "2 |") only valid at the last non-empty line
            if _PAGE_PIPE_RE.match(s) or _is_header_footer_line(s):
                lines[i] = ""
            else:
                break                   # last non-empty line is real content — stop
        elif _is_header_footer_line(s):
            lines[i] = ""              # subsequent footer line — remove it
        else:
            break                       # real content ends here — stop

    # Strip leading and trailing blank/whitespace-only lines left by layout=True
    while lines and not lines[0].strip():
        lines.pop(0)
    while lines and not lines[-1].strip():
        lines.pop()

    return "\n".join(lines)


# Test on the inspect page
raw   = raw_pages[PAGE_TO_INSPECT]
clean = strip_header_footer(raw)

# Compare content (stripped) so trailing whitespace differences don't show as false positives
original_content = {l.strip() for l in raw.split("\n") if l.strip()}
cleaned_content  = {l.strip() for l in clean.split("\n") if l.strip()}
removed          = sorted(original_content - cleaned_content)

print("Lines removed:")
for line in removed:
    print(f"  [{line}]")

Lines removed:
  [Fineotex Chemical Limited]
  [May 18, 2026]
  [Page 2 of 20]


In [229]:
# Confirm the cleaned page looks correct — no header, body text intact
print(f"=== Cleaned page {CONTENT_START_PAGE + PAGE_TO_INSPECT} ===")
print(clean)

=== Cleaned page 3 ===
                                                                                  
                                                                                  
                                                                                  


                                                                                  
                                                                                  
                                                                                  
                                                                                  
          Moderator:     Ladies and gentlemen, good day, and welcome to the Fineotex Chemical Limited Q4 and FY '26
                         Earnings Conference Call. As a reminder, all participant lines will be in the listen-only mode,
                         and there will be an opportunity for you to ask questions after the presentation concludes. Should
                         you need

In [230]:
# Apply to all content pages and check another page
clean_pages = [strip_header_footer(p) for p in raw_pages]

CHECK_PAGE = 3   # 0-indexed among content pages — change to inspect any page
print(f"=== Cleaned page {CONTENT_START_PAGE + CHECK_PAGE} ===")
print(clean_pages[CHECK_PAGE])

=== Cleaned page 6 ===
                                                                                  
                                                                                  
                                                                                  


                                                                                  
                         These measures have started yielding encouraging results, including improvement in EBITDA
                         margins. Importantly, even post-acquisition and through the ongoing expansion initiatives, the
                                                                                  
                         company continues to maintain a strong financial position, enabling us to remain focused on
                         both organic and inorganic growth opportunities going forward.
                                                                                  
                         Going forwar

---
## Step 4 — Concatenate into one clean transcript string

Join all cleaned page texts. A speaker turn that starts at the bottom of one page and continues on the next is naturally joined — no special handling needed.

In [231]:
full_text = "\n".join(clean_pages)

print(f"Total characters : {len(full_text):,}")
print(f"Total words      : {len(full_text.split()):,}")
print()
print("First 2000 characters:")
print(full_text[:2000])
# print(full_text)

Total characters : 113,707
Total words      : 9,170

First 2000 characters:
                                                                                  
                                                                                  
                                                                                  


                                                                                  
                                                                                  
                                                                                  
                                                                                  
          Moderator:     Ladies and gentlemen, good day, and welcome to the Fineotex Chemical Limited Q4 and FY '26
                         Earnings Conference Call. As a reminder, all participant lines will be in the listen-only mode,
                         and there will be an opportunity for you to ask questions after the presentatio

---
## Step 5 — Detect all speaker names from the transcript

**Approach: paragraph boundary detection**

Every speaker turn opens a new paragraph (preceded by a blank line or document start). Only a paragraph's first non-blank line can be a speaker name — this eliminates false positives that appear mid-comment.

**Three-layer filter:**
1. **Paragraph boundary** — only inspect the first non-blank line of each paragraph
2. **Title Case + colon pattern** — `[A-Z][a-z]+` rejects ALL-CAPS abbreviations (`EBITDA:`, `Q:`) and lowercase sentence starters; optional leading initials `J.` or `R.J.` allowed before first proper word
3. **Single-word blocklist** — rejects known section labels (`Note:`, `Disclaimer:`) while keeping single-word analyst names (`Richa:`) and `Moderator:`

**What this handles that the old approach did not:**
- Names with leading initials like `J. Lakshmana Rao:` — `(?:[A-Z]\.\s+){0,2}` prefix
- Single-word analyst names like `Richa:` — not in blocklist, kept
- One-question analysts (appear only once) — no recurrence filter needed
- Both indentation styles — `^\s*` accepts 0 spaces (Asian Paints) and 10+ spaces (Fineotex)

In [232]:
# Paragraph-boundary speaker pattern:
#   ^\s*                  — optional indent (handles 0-space Asian Paints and 10-space Fineotex)
#   (?:[A-Z]\.\s+){0,2}  — optional leading initials: 'J. ' or 'R.J. ' (0, 1, or 2 initials)
#   [A-Z][a-z]+           — first proper word: Title Case (rejects ALL-CAPS, rejects 'Q')
#   (...){0,3}            — up to 3 more words (allow trailing dots for initials mid-name)
#   \s*:                  — colon, optional space before (handles 'Name :' and 'Name:')
_PARA_SPEAKER_RE = re.compile(
    r"^\s*((?:[A-Z]\.\s+){0,2}[A-Z][a-z]+(?:\s+[A-Z][a-z.]*){0,3})\s*:"
)

# Single-word patterns that can appear at paragraph boundaries but are NOT speaker names
_SPEAKER_BLOCKLIST = {
    "Note", "Disclaimer", "Background", "Summary", "Conclusion",
    "Important", "Update", "Result", "Overview", "Outlook",
    "Please", "Date", "Venue", "Time", "Subject", "Dear",
    "Thanks", "Regards", "Encl", "Sir", "Madam", "Yours"
}


def is_valid_speaker(name):
    """Reject single-word blocklist entries; allow everything else including single-word names."""
    words = name.split()
    if len(words) == 1 and words[0] in _SPEAKER_BLOCKLIST:
        return False
    return True


def in_roster(name, roster_names):
    """Exact match OR single-word last-name fallback for transcript format differences.

    The last-name fallback handles cases where the transcript uses only a last name
    (e.g. 'Jeyamurugan' in transcript vs 'R.J. Jeyamurugan' in roster).

    Multi-word names (e.g. 'Chirag Jain') require exact match — no last-name fallback.
    This prevents false positives when different people share a last name
    (e.g. 'Chirag Jain' must not match 'Yashpal Jain' in the management roster).
    """
    if name in roster_names:
        return True
    words = name.split()
    if len(words) == 1:                          # single-word transcript name only
        last = words[0].lower()
        if len(last) >= 4:
            return any(m.split()[-1].lower() == last for m in roster_names)
    return False


# kept for backwards compat — used in cell-20
in_mgmt_roster = in_roster


# Split full text into paragraphs on 2+ consecutive blank/whitespace-only lines
paragraphs = re.split(r'(?:\n[ \t]*){2,}', full_text)

detected = Counter()
for para in paragraphs:
    non_blank = [l for l in para.split('\n') if l.strip()]
    if not non_blank:
        continue
    first_line = non_blank[0]          # first non-blank line of this paragraph
    m = _PARA_SPEAKER_RE.match(first_line)
    if m:
        name = m.group(1).strip()
        if is_valid_speaker(name):
            detected[name] += 1

print(f"Speaker names detected ({len(detected)} unique):")
print()
print(f"  {'turns':>6}  {'role':<16} name")
print("  " + "-" * 50)
for name, count in detected.most_common():
    if re.match(r"^(moderator|operator)$", name, re.I):
        roster_label = "[moderator]"
    elif name in moderator_names:           # exact match only for moderator
        roster_label = "[moderator]"
    elif in_roster(name, mgmt_names):
        roster_label = "[management]"
    else:
        roster_label = ""
    print(f"  {count:>6}  {roster_label:<16} {name}")

Speaker names detected (16 unique):

   turns  role             name
  --------------------------------------------------
      49  [management]     Sanjay Tibrewala
      18  [moderator]      Moderator
       9                   Darshil Jhaveri
       9                   Rohit Ohri
       6                   Pritesh Chheda
       6                   Nalin Shah
       5                   Amit Mehendale
       5                   Anupam Agarwal
       4                   Sunil Jain
       4                   Vinay Nadkarni
       3                   Karan Kamdar
       2  [management]     Aarti Jhunjhunwala
       2                   Keshv Garg
       1  [management]     Arindam Choudhuri
       1  [management]     Yusuf Contractor
       1                   Keshav Garg


In [233]:
# Inspect the detected list above — if any false positives slipped through the blocklist, add them here
EXCLUDE_NAMES = set()   # e.g. {"Overview"} if a section heading was picked up

speaker_names = sorted(
    name for name in detected if name not in EXCLUDE_NAMES
)

print(f"Final speaker roster ({len(speaker_names)} names): {speaker_names}")

Final speaker roster (16 names): ['Aarti Jhunjhunwala', 'Amit Mehendale', 'Anupam Agarwal', 'Arindam Choudhuri', 'Darshil Jhaveri', 'Karan Kamdar', 'Keshav Garg', 'Keshv Garg', 'Moderator', 'Nalin Shah', 'Pritesh Chheda', 'Rohit Ohri', 'Sanjay Tibrewala', 'Sunil Jain', 'Vinay Nadkarni', 'Yusuf Contractor']


---
## Step 6 — Split text by speaker names → turns

Build one regex from the full roster and split the clean text at every `Name:` occurrence.  
Each segment = one complete turn (the full comment, with paragraph spacing preserved).

In [234]:
# Longest names first to avoid partial matches (e.g. "Aarti" before "Aarti Jhunjhunwala")
name_alts = "|".join(re.escape(n) for n in sorted(speaker_names, key=len, reverse=True))

# ^\s*   — any indent (0-space or 10-space formats)
# \s*:   — optional space before colon (handles 'Name :' Asian Paints style)
_SPLIT_RE = re.compile(rf"^\s*({name_alts})\s*:", re.MULTILINE)

parts = _SPLIT_RE.split(full_text)
# split() with a capture group → [pre_text, name1, block1, name2, block2, ...]

print(f"Parts from split: {len(parts)}")
print(f"Expected turns  : {(len(parts) - 1) // 2}")

if parts[0].strip():
    print(f"\nPre-first-speaker text ({len(parts[0].strip())} chars):")
    print(parts[0].strip()[:300])

Parts from split: 251
Expected turns  : 125


In [235]:
_MOD_RE = re.compile(r"^(moderator|operator)$", re.I)


def is_management_speaker(speaker, mgmt_names):
    return in_roster(speaker, mgmt_names)


def is_moderator_speaker(speaker, moderator_names):
    """True if speaker is the generic 'Moderator'/'Operator' keyword OR listed on page 2 roster.

    Uses EXACT match only for named moderators — no last-name fallback.
    Last-name matching is intentionally omitted here because common Indian surnames
    (e.g. 'Jain') would cause false positives when management or analysts share the name.
    """
    if _MOD_RE.match(speaker.strip()):
        return True
    return speaker in moderator_names   # exact match only


def build_turns(parts, mgmt_names, moderator_names):
    def role(speaker):
        if is_moderator_speaker(speaker, moderator_names):
            return "moderator"
        if is_management_speaker(speaker, mgmt_names):
            return "management"
        return "analyst"

    turns = []
    i = 1
    while i + 1 < len(parts):
        speaker = parts[i].strip()
        comment = parts[i + 1].strip()
        if comment:
            turns.append({"speaker": speaker, "role": role(speaker), "comment": comment})
        i += 2
    return turns


turns = build_turns(parts, mgmt_names, moderator_names)

role_counts = Counter(t["role"] for t in turns)
print(f"Total turns : {len(turns)}")
for role, count in role_counts.most_common():
    print(f"  {role:<12} {count}")

Total turns : 125
  analyst      54
  management   53
  moderator    18


---
## Step 7 — Turn-level validation (intermediate)

Intermediate view: raw turn sequence before session grouping. Use this to verify speaker detection and role assignment. The cells below (overview table, validation report) are debugging aids — they confirm Steps 1–6 are working before producing the final Stage 0 output in Step 9.

In [236]:
# Overview table — all turns
print(f"  {'#':<5} {'role':<12} {'speaker':<30} {'words'}")
print("  " + "-" * 62)
for i, t in enumerate(turns):
    words = len(t["comment"].split())
    print(f"  {i:<5} {t['role']:<12} {t['speaker']:<30} {words}")

  #     role         speaker                        words
  --------------------------------------------------------------
  0     moderator    Moderator                      100
  1     management   Aarti Jhunjhunwala             414
  2     management   Arindam Choudhuri              296
  3     management   Yusuf Contractor               235
  4     management   Sanjay Tibrewala               283
  5     moderator    Moderator                      30
  6     analyst      Amit Mehendale                 69
  7     management   Sanjay Tibrewala               161
  8     analyst      Amit Mehendale                 43
  9     management   Sanjay Tibrewala               183
  10    analyst      Amit Mehendale                 24
  11    management   Sanjay Tibrewala               166
  12    analyst      Amit Mehendale                 7
  13    moderator    Moderator                      19
  14    analyst      Amit Mehendale                 2
  15    moderator    Moderator                

In [237]:
# Inspect any single turn in full — change TURN_INDEX
TURN_INDEX = 0

t = turns[TURN_INDEX]
print(f"Speaker : {t['speaker']}")
print(f"Role    : {t['role']}")
print(f"Words   : {len(t['comment'].split())}")
print()
print(t["comment"])

Speaker : Moderator
Role    : moderator
Words   : 100

Ladies and gentlemen, good day, and welcome to the Fineotex Chemical Limited Q4 and FY '26
                         Earnings Conference Call. As a reminder, all participant lines will be in the listen-only mode,
                         and there will be an opportunity for you to ask questions after the presentation concludes. Should
                         you need assistance during this conference call, please signal an operator by pressing star then
                                                                                  
                         zero on your touchtone phone. Please note that this conference is being recorded.
                                                                                  
                         I now hand the conference over to Ms. Aarti Jhunjhunwala, Fineotex Chemical Limited. Thank
                         you, and over to you, ma'am. Aarti ma'am over to you.


In [238]:
# Print the first 5 turns formatted
for t in turns[:5]:
    print("=" * 70)
    print(f"[{t['role'].upper()}]  {t['speaker']}")
    print("=" * 70)
    print(t["comment"])
    print()

[MODERATOR]  Moderator
Ladies and gentlemen, good day, and welcome to the Fineotex Chemical Limited Q4 and FY '26
                         Earnings Conference Call. As a reminder, all participant lines will be in the listen-only mode,
                         and there will be an opportunity for you to ask questions after the presentation concludes. Should
                         you need assistance during this conference call, please signal an operator by pressing star then
                                                                                  
                         zero on your touchtone phone. Please note that this conference is being recorded.
                                                                                  
                         I now hand the conference over to Ms. Aarti Jhunjhunwala, Fineotex Chemical Limited. Thank
                         you, and over to you, ma'am. Aarti ma'am over to you.

[MANAGEMENT]  Aarti Jhunjhunwala
Good morning, eve

In [239]:
import os

W = 70

def section(title):
    print(f"\n{title}")
    print("─" * W)

print("═" * W)
print(f"VALIDATION REPORT — {os.path.basename(PDF_PATH)}")
print("═" * W)

# ── [1] Page 2 roster ────────────────────────────────────────────────
section("[1] PAGE 2 ROSTER")
print(f"  Management ({len(mgmt_names)}):  " + (", ".join(sorted(mgmt_names)) or "(none found — check WARNING above)"))
print(f"  Moderator  ({len(moderator_names)}):  " + (", ".join(sorted(moderator_names)) or "(none listed — 'Moderator' keyword used in body)"))

# ── [2] Header / footer removal ──────────────────────────────────────
section("[2] HEADER / FOOTER STRIPPING")
all_removed = []
for raw, clean in zip(raw_pages, clean_pages):
    orig = {l.strip() for l in raw.split("\n") if l.strip()}
    kept = {l.strip() for l in clean.split("\n") if l.strip()}
    all_removed.extend(orig - kept)

removed_unique = sorted(set(all_removed))
print(f"  Total removed : {len(all_removed)} lines across {len(raw_pages)} content pages")
print(f"  Unique patterns ({len(removed_unique)}):")
for line in removed_unique[:10]:
    print(f"    [{line}]")
if len(removed_unique) > 10:
    print(f"    ... and {len(removed_unique) - 10} more")

# ── [3] Speaker roster with roles ────────────────────────────────────
section(f"[3] SPEAKER DETECTION  ({len(detected)} unique)")
print(f"  {'turns':>6}  {'role':<16} name")
print("  " + "─" * 46)
for name, count in detected.most_common():
    if re.match(r"^(moderator|operator)$", name, re.I):
        label = "[moderator]"
    elif name in moderator_names:           # exact match only for moderator
        label = "[moderator]"
    elif in_roster(name, mgmt_names):
        label = "[management]"
    else:
        label = ""
    print(f"  {count:>6}  {label:<16} {name}")

# ── [4] Role distribution ────────────────────────────────────────────
section("[4] ROLE DISTRIBUTION")
role_counts = Counter(t["role"] for t in turns)
for role_name, count in role_counts.most_common():
    bar = "█" * count
    print(f"  {role_name:<12} {count:>4}  {bar}")
print(f"  {'TOTAL':<12} {len(turns):>4}")

# ── [5] Full turn table ──────────────────────────────────────────────
section(f"[5] TURN TABLE  ({len(turns)} turns)")
print(f"  {'#':<5} {'role':<12} {'speaker':<30} {'words':>5}")
print("  " + "─" * 56)
for i, t in enumerate(turns):
    words = len(t["comment"].split())
    print(f"  {i:<5} {t['role']:<12} {t['speaker']:<30} {words:>5}")

print()
print("═" * W)

══════════════════════════════════════════════════════════════════════
VALIDATION REPORT — fineotex_chemical_Q4_FY26.pdf
══════════════════════════════════════════════════════════════════════

[1] PAGE 2 ROSTER
──────────────────────────────────────────────────────────────────────
  Management (4):  Aarti Jhunjhunwala, Arindam Choudhuri, Sanjay Tibrewala, Yusuf Contractor
  Moderator  (0):  (none listed — 'Moderator' keyword used in body)

[2] HEADER / FOOTER STRIPPING
──────────────────────────────────────────────────────────────────────
  Total removed : 57 lines across 19 content pages
  Unique patterns (21):
    [Fineotex Chemical Limited]
    [May 18, 2026]
    [Page 10 of 20]
    [Page 11 of 20]
    [Page 12 of 20]
    [Page 13 of 20]
    [Page 14 of 20]
    [Page 15 of 20]
    [Page 16 of 20]
    [Page 17 of 20]
    ... and 11 more

[3] SPEAKER DETECTION  (16 unique)
──────────────────────────────────────────────────────────────────────
   turns  role             name
  ────────

---
## Step 8 — Page boundaries and turns with char offsets

To produce `char_start`, `char_end`, `page_start`, `page_end` on each Chunk, we need to know where each speaker turn sits in the full concatenated string and which PDF page that maps to.

The split in Step 6 used `re.split()` which discards positional information. Here we use `finditer` on the same `_SPLIT_RE` to recover character positions for every turn.

In [ ]:
def build_page_boundaries(clean_pages, content_start_page):
    """
    Map each positional PDF page number to (char_start, char_end) in full_text.
    full_text = "\n".join(clean_pages), so each separator adds 1 char between pages.
    Page numbers are 1-indexed positional (by order in PDF file).
    """
    boundaries = {}
    offset = 0
    for idx, page_text in enumerate(clean_pages):
        page_num = content_start_page + idx
        boundaries[page_num] = (offset, offset + len(page_text))
        offset += len(page_text) + 1  # +1 for the "\n" separator from join
    return boundaries


def char_to_page(char_offset, page_boundaries):
    """Return the positional PDF page number that contains char_offset."""
    for pn in sorted(page_boundaries.keys()):
        start, end = page_boundaries[pn]
        if start <= char_offset < end:
            return pn
    return max(page_boundaries.keys())  # fallback: last page




def normalize_comment(raw_text):
    """Single turn = one continuous line. All newlines removed, spaces collapsed."""
    text = raw_text.replace('\n', ' ')
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()


page_boundaries = build_page_boundaries(clean_pages, CONTENT_START_PAGE)

print(f"Page boundaries ({len(page_boundaries)} content pages):")
for pn, (start, end) in list(page_boundaries.items())[:6]:
    print(f"  Page {pn:3d}: chars {start:7,} \u2013 {end:7,}  ({end - start:,} chars)")
if len(page_boundaries) > 6:
    print(f"  ... ({len(page_boundaries) - 6} more pages)")
print(f"\nTotal chars covered: {sum(e - s for s, e in page_boundaries.values()):,}")


def extract_turns_with_offsets(full_text, split_re, mgmt_names, moderator_names, page_boundaries):
    """
    Re-extract speaker turns using finditer to recover character positions.
    Returns list of dicts: {speaker, role, text, char_start, char_end, page_start, page_end}
    """
    matches = list(split_re.finditer(full_text))

    result = []
    for idx, m in enumerate(matches):
        speaker = m.group(1).strip()
        text_start = m.end()
        text_end = matches[idx + 1].start() if idx + 1 < len(matches) else len(full_text)
        normalized = normalize_comment(full_text[text_start:text_end])

        if not normalized:
            continue

        char_start = m.start()
        char_end = text_end
        page_start = char_to_page(char_start, page_boundaries)
        page_end = char_to_page(max(char_end - 1, char_start), page_boundaries)

        if is_moderator_speaker(speaker, moderator_names):
            role = 'moderator'
        elif is_management_speaker(speaker, mgmt_names):
            role = 'management'
        else:
            role = 'analyst'

        result.append({
            'speaker':    speaker,
            'role':       role,
            'text':       normalized,
            'char_start': char_start,
            'char_end':   char_end,
            'page_start': page_start,
            'page_end':   page_end,
        })

    return result


raw_turns_meta = extract_turns_with_offsets(
    full_text, _SPLIT_RE, mgmt_names, moderator_names, page_boundaries
)

print(f"\nTurns with char offsets: {len(raw_turns_meta)}")
print()
print(f"  {'#':<4} {'role':<12} {'speaker':<28} {'pg':>3}  {'char_start':>10}  {'char_end':>10}")
print("  " + "\u2500" * 68)
for i, t in enumerate(raw_turns_meta):
    print(f"  {i:<4} {t['role']:<12} {t['speaker']:<28} {t['page_start']:>3}  {t['char_start']:>10,}  {t['char_end']:>10,}")


---
## Step 9 — Session grouping → Stage 0 output (`List[Chunk]`)

Convert the raw turn sequence into `List[Chunk]` objects. The key design decision is **analyst-session grouping**: all turns belonging to one analyst's engagement (questions, moderator interruptions, management answers, follow-ups) are grouped into a single `QA_SESSION` chunk. This preserves the conversational context Stage 2 needs to extract guidance correctly.

**Chunk types:**
- `OPENING_REMARKS` — management turns before the first analyst, split at 600 words
- `QA_SESSION` — all turns for one analyst session until a different analyst or clean moderator transition
- `MANAGEMENT_SOLO` — standalone management turns mid-call (rare)

In [241]:
from dataclasses import dataclass, field
from enum import Enum
from typing import List, Optional


# ── Data models ───────────────────────────────────────────────────────────────

class TurnRole(str, Enum):
    MANAGEMENT = "management"
    ANALYST    = "analyst"
    MODERATOR  = "moderator"


class ChunkType(str, Enum):
    OPENING_REMARKS = "opening_remarks"
    QA_SESSION      = "qa_session"
    MANAGEMENT_SOLO = "management_solo"


@dataclass
class Turn:
    speaker:    str
    role:       TurnRole
    text:       str
    page_start: int
    page_end:   int
    char_start: int
    char_end:   int


@dataclass
class Chunk:
    chunk_id:        str
    chunk_type:      ChunkType
    speaker:         str
    analyst_speaker: Optional[str]
    role:            TurnRole
    page_start:      int
    page_end:        int
    char_start:      int
    char_end:        int
    word_count:      int
    text:            str
    turns:           List[Turn] = field(default_factory=list)


# ── Helpers ───────────────────────────────────────────────────────────────────

def make_turn_obj(t):
    return Turn(
        speaker=t['speaker'], role=TurnRole(t['role']), text=t['text'],
        page_start=t['page_start'], page_end=t['page_end'],
        char_start=t['char_start'], char_end=t['char_end'],
    )


def format_chunk_text(session_turns):
    parts = [f"{t['role'].title()} ({t['speaker']}): {t['text']}" for t in session_turns]
    return "\n\n".join(parts)


def make_chunk_obj(chunk_id, chunk_type, session_turns, analyst_speaker=None):
    mgmt_speaker = next(
        (t['speaker'] for t in session_turns if t['role'] == 'management'),
        session_turns[0]['speaker'],
    )
    text = format_chunk_text(session_turns)
    return Chunk(
        chunk_id=chunk_id, chunk_type=chunk_type, speaker=mgmt_speaker,
        analyst_speaker=analyst_speaker, role=TurnRole.MANAGEMENT,
        page_start=session_turns[0]['page_start'], page_end=session_turns[-1]['page_end'],
        char_start=session_turns[0]['char_start'], char_end=session_turns[-1]['char_end'],
        word_count=len(text.split()), text=text,
        turns=[make_turn_obj(t) for t in session_turns],
    )


OPENING_WORD_LIMIT = 600


def split_turn_by_paragraphs(turn_dict, word_limit):
    paras = [p.strip() for p in turn_dict['text'].split('\n\n') if p.strip()]
    groups, current, current_words = [], [], 0
    for para in paras:
        pw = len(para.split())
        if current_words + pw > word_limit and current:
            groups.append('\n\n'.join(current))
            current, current_words = [], 0
        current.append(para)
        current_words += pw
    if current:
        groups.append('\n\n'.join(current))
    if len(groups) <= 1:
        return [turn_dict]
    return [{**turn_dict, 'text': g} for g in groups]


# ── Session grouping ──────────────────────────────────────────────────────────

def group_into_chunks(raw_turns):
    chunks = []
    chunk_seq = 0

    first_analyst = next(
        (i for i, t in enumerate(raw_turns) if t['role'] == 'analyst'),
        len(raw_turns),
    )

    # ── Opening remarks ───────────────────────────────────────────────────────
    # Scan turns before the first analyst.
    # Moderator turns sandwiched between management turns stay in the opening
    # (e.g. "I now invite our MD for his remarks").
    # Moderator turns that trail at the END of the opening — after the last
    # management turn — are seeded into pending_mod so they prepend to the
    # first QA session, consistent with how all other between-session moderator
    # turns are handled.
    opening_pool = []
    mod_buffer = []   # holds moderator turns whose placement is not yet decided

    for t in raw_turns[:first_analyst]:
        if t['role'] == 'management':
            # A management turn confirms that buffered moderator turns before it
            # are within the opening section — commit them.
            opening_pool.extend(mod_buffer)
            mod_buffer = []
            opening_pool.extend(split_turn_by_paragraphs(t, OPENING_WORD_LIMIT))
        else:
            mod_buffer.append(t)

    # Whatever remains in mod_buffer is trailing moderator content after the last
    # management turn — send it to the first QA session, not to opening remarks.
    pending_mod = mod_buffer

    current_group, current_words = [], 0
    for sub in opening_pool:
        sw = len(sub['text'].split())
        if current_words + sw > OPENING_WORD_LIMIT and current_group:
            chunk_seq += 1
            chunks.append(make_chunk_obj(f"chunk_{chunk_seq:03d}", ChunkType.OPENING_REMARKS, current_group))
            current_group, current_words = [], 0
        current_group.append(sub)
        current_words += sw
    if current_group:
        chunk_seq += 1
        chunks.append(make_chunk_obj(f"chunk_{chunk_seq:03d}", ChunkType.OPENING_REMARKS, current_group))

    # ── Q&A sessions and management solo turns ────────────────────────────────
    i = first_analyst
    n = len(raw_turns)

    while i < n:
        t = raw_turns[i]

        if t['role'] == 'analyst':
            analyst_name = t['speaker']
            session = pending_mod + [t]
            pending_mod = []
            i += 1

            # local_mod buffers moderator turns whose placement is undecided.
            # Committed to session if the next non-moderator turn continues it;
            # moved to pending_mod if the next non-moderator turn is a new analyst.
            local_mod = []

            while i < n:
                curr = raw_turns[i]

                if curr['role'] == 'moderator':
                    local_mod.append(curr)
                    i += 1

                elif curr['role'] == 'analyst' and curr['speaker'] != analyst_name:
                    # Different analyst — local_mod is between-session
                    pending_mod = local_mod
                    local_mod = []
                    break

                else:
                    # Management or same analyst — local_mod is within-session
                    session.extend(local_mod)
                    local_mod = []
                    session.append(curr)
                    i += 1

            # End of transcript: flush remaining local_mod into this session
            session.extend(local_mod)

            chunk_seq += 1
            chunks.append(make_chunk_obj(
                f"chunk_{chunk_seq:03d}", ChunkType.QA_SESSION, session, analyst_name
            ))

        elif t['role'] == 'moderator':
            pending_mod.append(t)
            i += 1

        elif t['role'] == 'management':
            session = pending_mod + [t]
            pending_mod = []
            chunk_seq += 1
            chunks.append(make_chunk_obj(f"chunk_{chunk_seq:03d}", ChunkType.MANAGEMENT_SOLO, session))
            i += 1

    # Trailing moderator closing remarks with no following session
    if pending_mod and chunks:
        last = chunks[-1]
        extra = "\n\n" + "\n\n".join(
            f"{t['role'].title()} ({t['speaker']}): {t['text']}" for t in pending_mod
        )
        last.text += extra
        last.word_count = len(last.text.split())
        last.turns.extend(make_turn_obj(t) for t in pending_mod)
        last.page_end = pending_mod[-1]['page_end']
        last.char_end = pending_mod[-1]['char_end']

    return chunks


# ── Run ───────────────────────────────────────────────────────────────────────

chunks = group_into_chunks(raw_turns_meta)

type_counts = Counter(c.chunk_type.value for c in chunks)
print(f"Stage 0 output: {len(chunks)} chunks")
print()
for ct, cnt in type_counts.most_common():
    print(f"  {ct:<22} {cnt}")
print()
print(f"  {'chunk_id':<12} {'type':<20} {'speaker':<22} {'analyst':<24} {'pages':>8}  {'words':>6}  {'turns':>6}")
print("  " + "─" * 106)
for c in chunks:
    analyst = c.analyst_speaker or "—"
    pages = f"{c.page_start}–{c.page_end}" if c.page_start != c.page_end else str(c.page_start)
    print(f"  {c.chunk_id:<12} {c.chunk_type.value:<20} {c.speaker:<22} {analyst:<24} {pages:>8}  {c.word_count:>6}  {len(c.turns):>6}")

Stage 0 output: 15 chunks

  qa_session             12
  opening_remarks        3

  chunk_id     type                 speaker                analyst                     pages   words   turns
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────
  chunk_001    opening_remarks      Aarti Jhunjhunwala     —                             3–4     519       2
  chunk_002    opening_remarks      Arindam Choudhuri      —                             4–5     537       2
  chunk_003    opening_remarks      Sanjay Tibrewala       —                             5–6     286       1
  chunk_004    qa_session           Sanjay Tibrewala       Amit Mehendale                6–7     732      10
  chunk_005    qa_session           Sanjay Tibrewala       Pritesh Chheda                7–9     857      13
  chunk_006    qa_session           Sanjay Tibrewala       Darshil Jhaveri              9–11     860      18
  chunk_007    qa_session           Sanjay Ti

In [242]:
# Inspect any single chunk — change CHUNK_IDX to explore different chunks
CHUNK_IDX = 0

c = chunks[CHUNK_IDX]
W = 72
print("═" * W)
print(f"  {c.chunk_id}  [{c.chunk_type.value}]")
print(f"  Speaker      : {c.speaker}")
print(f"  Analyst      : {c.analyst_speaker or '—'}")
print(f"  Pages        : {c.page_start} – {c.page_end}")
print(f"  Char offsets : {c.char_start:,} – {c.char_end:,}")
print(f"  Word count   : {c.word_count}  {'(will be windowed by Stage 2)' if c.word_count > 600 else '(fits in one extraction pass)'}")
print(f"  Turns        : {len(c.turns)}")
print("─" * W)
print()
snippet = c.text[:2000]
print(snippet)
if len(c.text) > 2000:
    print(f"\n... [{len(c.text) - 2000:,} more chars]")
print()
print("─" * W)
print("TURNS:")
for j, turn in enumerate(c.turns):
    words = len(turn.text.split())
    print(f"  [{j}] {turn.role.value:<12} {turn.speaker:<28} pages {turn.page_start}–{turn.page_end}  ({words} words)")
print("═" * W)

════════════════════════════════════════════════════════════════════════
  chunk_001  [opening_remarks]
  Speaker      : Aarti Jhunjhunwala
  Analyst      : —
  Pages        : 3 – 4
  Char offsets : 0 – 7,513
  Word count   : 519  (fits in one extraction pass)
  Turns        : 2
────────────────────────────────────────────────────────────────────────

Moderator (Moderator): Ladies and gentlemen, good day, and welcome to the Fineotex Chemical Limited Q4 and FY '26
                         Earnings Conference Call. As a reminder, all participant lines will be in the listen-only mode,
                         and there will be an opportunity for you to ask questions after the presentation concludes. Should
                         you need assistance during this conference call, please signal an operator by pressing star then
                                                                                  
                         zero on your touchtone phone. Please note that this confe

In [243]:
W = 80

for c in chunks:
    # ── Chunk header ──────────────────────────────────────────────────────────
    print("█" * W)
    print(f"  {c.chunk_id}  ·  {c.chunk_type.value.upper()}")
    print(f"  Speaker   : {c.speaker}")
    if c.analyst_speaker:
        print(f"  Analyst   : {c.analyst_speaker}")
    print(f"  Pages     : {c.page_start}–{c.page_end}   |   Words: {c.word_count}   |   Turns: {len(c.turns)}")
    print("─" * W)

    # ── Turn breakdown (compact) ──────────────────────────────────────────────
    for j, turn in enumerate(c.turns):
        role_tag = {"management": "MGMT", "analyst": "ANA ", "moderator": "MOD "}[turn.role.value]
        print(f"  [{j}] {role_tag}  {turn.speaker}")

    print("─" * W)

    # ── Full text ─────────────────────────────────────────────────────────────
    print()
    print(c.text)
    print()
    print()

████████████████████████████████████████████████████████████████████████████████
  chunk_001  ·  OPENING_REMARKS
  Speaker   : Aarti Jhunjhunwala
  Pages     : 3–4   |   Words: 519   |   Turns: 2
────────────────────────────────────────────────────────────────────────────────
  [0] MOD   Moderator
  [1] MGMT  Aarti Jhunjhunwala
────────────────────────────────────────────────────────────────────────────────

Moderator (Moderator): Ladies and gentlemen, good day, and welcome to the Fineotex Chemical Limited Q4 and FY '26
                         Earnings Conference Call. As a reminder, all participant lines will be in the listen-only mode,
                         and there will be an opportunity for you to ask questions after the presentation concludes. Should
                         you need assistance during this conference call, please signal an operator by pressing star then
                                                                                  
                        

---
## Acceptance Tests

Run these on all target transcripts (switch `PDF_PATH` in the Setup cell and re-run from the top) before moving to Stage 1.

- **Test 1** — Speaker role detection: no management speaker misclassified as analyst
- **Test 2** — Chunk count sanity: 10–200 chunks for a typical transcript
- **Test 3** — QA_SESSION coverage: ≥60% of chunks; opening remarks confined to the start
- **Test 4** — GT item containment **[critical, must be 100%]**: every GT passage appears in at least one chunk's `text`

In [244]:
import os

print("═" * 72)
print(f"ACCEPTANCE TESTS — {os.path.basename(PDF_PATH)}")
print("═" * 72)


# ── Test 1 — Speaker role detection ──────────────────────────────────────────
print("\nTEST 1 — Speaker Role Detection")
print("─" * 72)

seen = {}
for t in raw_turns_meta:
    if t['speaker'] not in seen:
        seen[t['speaker']] = t['role']

print(f"  {'role':<14} speaker")
print("  " + "─" * 48)
t1_failures = []
for speaker, role in sorted(seen.items(), key=lambda x: (x[1], x[0])):
    flag = ""
    if role == 'analyst' and is_management_speaker(speaker, mgmt_names):
        flag = "  ← FAIL: management misclassified as analyst"
        t1_failures.append(speaker)
    print(f"  {role:<14} {speaker}{flag}")

print()
print("  >>> MANUALLY VERIFY: confirm all [management] names above match company roster")
if t1_failures:
    print(f"  AUTOMATED RESULT: FAIL — {t1_failures} misclassified as analyst")
else:
    print("  AUTOMATED RESULT: PASS (no management speaker found in analyst role)")


# ── Test 2 — Chunk count sanity ───────────────────────────────────────────────
print("\nTEST 2 — Chunk Count Sanity")
print("─" * 72)
total_chunks = len(chunks)
print(f"  PDF content pages : {len(raw_pages)}")
print(f"  Total chunks      : {total_chunks}")
if total_chunks < 10:
    print("  RESULT: FAIL — fewer than 10 chunks (speaker regex likely too strict)")
elif total_chunks > 200:
    print("  RESULT: FAIL — more than 200 chunks (regex may be matching non-headers)")
else:
    print(f"  RESULT: PASS — {total_chunks} chunks is in expected range (10–200)")


# ── Test 3 — QA_SESSION coverage ─────────────────────────────────────────────
print("\nTEST 3 — QA_SESSION Coverage")
print("─" * 72)
qa_chunks      = [c for c in chunks if c.chunk_type == ChunkType.QA_SESSION]
opening_chunks = [c for c in chunks if c.chunk_type == ChunkType.OPENING_REMARKS]
solo_chunks    = [c for c in chunks if c.chunk_type == ChunkType.MANAGEMENT_SOLO]
qa_pct = len(qa_chunks) / len(chunks) * 100 if chunks else 0
last_opening_seq = max(
    (int(c.chunk_id.replace("chunk_", "")) for c in opening_chunks), default=0
)

print(f"  OPENING_REMARKS : {len(opening_chunks):>3}  (IDs: {', '.join(c.chunk_id for c in opening_chunks)})")
print(f"  QA_SESSION      : {len(qa_chunks):>3}  ({qa_pct:.0f}% of total)")
print(f"  MANAGEMENT_SOLO : {len(solo_chunks):>3}")

t3_qa = qa_pct >= 60
t3_order = last_opening_seq <= 10
if t3_qa and t3_order:
    print(f"  RESULT: PASS — {qa_pct:.0f}% QA_SESSION (≥60%); opening remarks at start")
else:
    if not t3_qa:
        print(f"  RESULT: FAIL — only {qa_pct:.0f}% QA_SESSION (need ≥60%)")
    if not t3_order:
        print(f"  RESULT: FAIL — OPENING_REMARKS extends to chunk_{last_opening_seq:03d} (expected within first 10)")


# ── Test 4 — GT item chunk containment [CRITICAL] ────────────────────────────
print("\nTEST 4 — GT Item Chunk Containment [must be 100%]")
print("─" * 72)

GT_PATHS = {
    "asian_paints":         "../data/asian_paints_Q4_FY26_ground_truth_v3.txt",
    "fineotex_chemical":    "../data/fineotex_chemical_Q4_FY26_ground_truth_v1.txt",
    "sandhar_technologies": "../data/sandhar_technologies_Q4_FY26_ground_truth_v1.txt",
    "mold-tek_packaging":   "../data/mold-tek_packaging_Q4_FY26_ground_truth_v1.txt",
}


def parse_gt_passages(gt_path):
    """Extract passage values from GT file (handles both quoted and unquoted formats)."""
    if not os.path.exists(gt_path):
        return []
    text = open(gt_path).read()
    passages = [m.group(1).strip() for m in re.finditer(r'passage\s*:\s*"(.*?)"', text, re.S)]
    if not passages:  # fallback for unquoted format
        passages = [m.group(1).strip() for m in re.finditer(r'^passage\s*:\s*(.+?)(?=^\w|\Z)', text, re.S | re.M)]
    return passages


pdf_key = next((k for k in GT_PATHS if k in os.path.basename(PDF_PATH).lower()), None)

if pdf_key is None:
    print("  WARNING: No GT file matches the current PDF.")
    print("  Switch PDF_PATH to asian_paints, fineotex_chemical, sandhar_technologies, or mold-tek_packaging")
else:
    gt_path = GT_PATHS[pdf_key]
    passages = parse_gt_passages(gt_path)
    print(f"  GT file  : {os.path.basename(gt_path)}")
    print(f"  GT items : {len(passages)}")
    print()

    chunk_texts = [(c.chunk_id, " ".join(c.text.split())) for c in chunks]
    passed = failed = 0

    for idx, passage in enumerate(passages):
        norm = " ".join(passage.split())
        found = [cid for cid, ct in chunk_texts if norm in ct]
        if found:
            print(f"  [PASS] GT item {idx + 1}: found in {found}")
            passed += 1
        else:
            print(f"  [FAIL] GT item {idx + 1}: NOT FOUND IN ANY CHUNK")
            print(f"         Passage (first 120 chars): {passage[:120]}...")
            failed += 1

    print()
    if failed == 0:
        print(f"  RESULT: PASS — all {len(passages)} GT passages found in chunks (100%)")
    else:
        print(f"  RESULT: FAIL — {failed}/{len(passages)} GT passages not found")
        print("          Stage 0 is BLOCKED until this is 100%")

print()
print("═" * 72)

════════════════════════════════════════════════════════════════════════
ACCEPTANCE TESTS — fineotex_chemical_Q4_FY26.pdf
════════════════════════════════════════════════════════════════════════

TEST 1 — Speaker Role Detection
────────────────────────────────────────────────────────────────────────
  role           speaker
  ────────────────────────────────────────────────
  analyst        Amit Mehendale
  analyst        Anupam Agarwal
  analyst        Darshil Jhaveri
  analyst        Karan Kamdar
  analyst        Keshav Garg
  analyst        Keshv Garg
  analyst        Nalin Shah
  analyst        Pritesh Chheda
  analyst        Rohit Ohri
  analyst        Sunil Jain
  analyst        Vinay Nadkarni
  management     Aarti Jhunjhunwala
  management     Arindam Choudhuri
  management     Sanjay Tibrewala
  management     Yusuf Contractor
  moderator      Moderator

  >>> MANUALLY VERIFY: confirm all [management] names above match company roster
  AUTOMATED RESULT: PASS (no management spe

In [245]:
pdf.close()
print("Done.")

Done.
